In [ ]:
!wget ftp://ftp.irisa.fr/local/texmex/corpus/sift.tar.gz
!tar -xzf sift.tar.gz

--2026-06-17 19:15:59--  ftp://ftp.irisa.fr/local/texmex/corpus/sift.tar.gz
           => ‘sift.tar.gz’
Resolving ftp.irisa.fr (ftp.irisa.fr)... 131.254.254.45, 2001:660:7303:254::45
Connecting to ftp.irisa.fr (ftp.irisa.fr)|131.254.254.45|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /local/texmex/corpus ... done.
==> SIZE sift.tar.gz ... 168280445
==> PASV ... done.    ==> RETR sift.tar.gz ... done.
Length: 168280445 (160M) (unauthoritative)

sift.tar.gz         100%[===================>] 160.48M  22.2MB/s    in 8.4s    

2026-06-17 19:16:10 (19.2 MB/s) - ‘sift.tar.gz’ saved [168280445]



In [ ]:
# ============================================================
# LSH Auto-Tuner (Grid Search) για το Πλάτος Κάδου (w)
# ============================================================

import numpy as np
import time
from pyspark.sql import SparkSession

# ==== 1. ΠΑΡΑΜΕΤΡΟΙ ====
W_FACTORS_TO_TEST = [0.5, 1.0, 1.5, 2.0, 3.0]
EPSILON   = 200.0
L_TABLES  = 50
K_FUNCS   = 4
CHUNK_SIZE = 1000
D_DIM     = 128
REC_LEN   = 4 + D_DIM * 4
R_LIMIT   = 100000
S_LIMIT   = 1000
R_PATH    = "sift/sift_base.fvecs"
S_PATH    = "sift/sift_query.fvecs"

# ==== 2. Spark ====
spark = SparkSession.builder \
    .appName("LSH_Tune_W") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.python.worker.memory", "2g") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "1g") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")
print(f"Spark {spark.version} | Πυρήνες: {sc.defaultParallelism}")

# ==== 3. Συναρτήσεις ====
def parse_fvecs(record):
    return np.frombuffer(bytes(record[4:]), dtype=np.float32).copy()

def generate_lsh_model(d, L, K, w, seed=42):
    np.random.seed(seed)
    A = [np.random.randn(K, d).astype(np.float32) for _ in range(L)]
    B = [np.random.uniform(0, w, K).astype(np.float32) for _ in range(L)]
    return A, B

def build_bucket_index(S_dict, A, B, w, L):
    buckets = {}
    for s_id, s_vec in S_dict.items():
        for l in range(L):
            proj = np.dot(A[l], s_vec) + B[l]
            key  = (l, tuple(np.floor(proj / w).astype(np.int32).tolist()))
            buckets.setdefault(key, []).append(s_id)
    return buckets

def _process_chunk(ids, vecs, A, B, w, L, s_bkts, s_vecs, eps_sq):
    M = np.stack(vecs)
    candidates = [set() for _ in range(len(ids))]
    for l in range(L):
        projs   = M @ A[l].T + B[l]
        buckets = np.floor(projs / w).astype(np.int32)
        for i in range(len(ids)):
            key = (l, tuple(buckets[i].tolist()))
            if key in s_bkts:
                candidates[i].update(s_bkts[key])
    for i, r_id in enumerate(ids):
        r = M[i]
        for s_id in candidates[i]:
            diff = r - s_vecs[s_id]
            if float(np.dot(diff, diff)) <= eps_sq:
                yield (r_id, s_id)

def lsh_map_chunked(iterator, A_bc, B_bc, w, L, Sb_bc, Sv_bc, eps):
    A = A_bc.value; B = B_bc.value
    sb = Sb_bc.value; sv = Sv_bc.value
    eps_sq = eps ** 2
    ids, vecs = [], []
    for r_id, r_vec in iterator:
        ids.append(r_id); vecs.append(r_vec)
        if len(ids) >= CHUNK_SIZE:
            yield from _process_chunk(ids, vecs, A, B, w, L, sb, sv, eps_sq)
            ids, vecs = [], []
    if ids:
        yield from _process_chunk(ids, vecs, A, B, w, L, sb, sv, eps_sq)

# ==== 4. Φόρτωση δεδομένων ====
print(f"\nΦόρτωση {R_LIMIT:,}×{S_LIMIT:,} dataset...")
N_PARTS = max(sc.defaultParallelism * 4, 20)

rdd_R_kv = (sc.binaryRecords(R_PATH, REC_LEN)
              .map(parse_fvecs).zipWithIndex()
              .filter(lambda x: x[1] < R_LIMIT)
              .map(lambda x: (f"R_{x[1]}", x[0]))
              .repartition(N_PARTS)
              .cache())

S_dict = dict(sc.binaryRecords(S_PATH, REC_LEN)
                .map(parse_fvecs).zipWithIndex()
                .filter(lambda x: x[1] < S_LIMIT)
                .map(lambda x: (f"S_{x[1]}", x[0]))
                .collect())

R_samp = dict(rdd_R_kv.collect())
S_samp = S_dict

# ==== 5. Υπολογισμός Ground Truth (ΜΙΑ ΦΟΡΑ) ====
print("\nΥπολογισμός Baseline (ground truth)...")
eps_sq = EPSILON ** 2
t_gt = time.time()
gt_pairs = set()
for r_id, r_vec in R_samp.items():
    for s_id, s_vec in S_samp.items():
        d = r_vec - s_vec
        if float(np.dot(d, d)) <= eps_sq:
            gt_pairs.add((r_id, s_id))
t_baseline = time.time() - t_gt
print(f"  Ground truth: {len(gt_pairs):,} ζευγάρια (Χρόνος Baseline: {t_baseline:.2f}s)")

# ==== 6. Loop επί w_factors ====
results = []

for w_fac in W_FACTORS_TO_TEST:
    W_WIDTH = EPSILON * w_fac

    print(f"\n{'='*60}")
    print(f"  ΕΛΕΓΧΟΣ: w_factor={w_fac}x | w={W_WIDTH} | ε={EPSILON} | L={L_TABLES}")
    print(f"{'='*60}")

    A_mat, B_mat = generate_lsh_model(D_DIM, L_TABLES, K_FUNCS, W_WIDTH)
    sb_local     = build_bucket_index(S_dict, A_mat, B_mat, W_WIDTH, L_TABLES)

    A_bc  = sc.broadcast(A_mat)
    B_bc  = sc.broadcast(B_mat)
    Sv_bc = sc.broadcast(S_dict)
    Sb_bc = sc.broadcast(sb_local)

    t0 = time.time()
    _L, _w, _e = L_TABLES, W_WIDTH, EPSILON
    matches = (rdd_R_kv
               .mapPartitions(lambda it,
                    _A=A_bc, _B=B_bc, _Sb=Sb_bc, _Sv=Sv_bc,
                    _l=_L, _ww=_w, _ee=_e:
                    lsh_map_chunked(it, _A, _B, _ww, _l, _Sb, _Sv, _ee))
               .collect())
    t_lsh = time.time() - t0

    ms     = set(matches)
    approx = {p for p in ms if p[0] in R_samp and p[1] in S_samp}
    recall = len(gt_pairs & approx) / len(gt_pairs) if gt_pairs else 1.0
    status = "✓" if recall >= 0.95 else ("~" if recall >= 0.90 else "✗")

    print(f"  LSH ζευγάρια: {len(matches):,} | Χρόνος LSH: {t_lsh:.2f}s | "
          f"Recall: {recall*100:.1f}% {status}")
    print(f"  Speedup vs baseline: {t_baseline/t_lsh:.1f}x")

    results.append((w_fac, W_WIDTH, t_lsh, t_baseline, recall, len(matches)))

    with open("results_W_tuning.txt", "a") as f:
        f.write(f"w_fac={w_fac:<4} w={W_WIDTH:<5} L={L_TABLES} K={K_FUNCS} eps={EPSILON} | "
                f"lsh={t_lsh:.2f}s baseline={t_baseline:.2f}s "
                f"recall={recall*100:.1f}% pairs={len(matches)}\n")

    A_bc.destroy(); B_bc.destroy(); Sv_bc.destroy(); Sb_bc.destroy()

# ==== 7. Σύνοψη ====
print(f"\n{'='*72}")
print(f"  {'w_factor':>8} | {'w':>5} | {'LSH(s)':>8} | {'Base(s)':>8} | "
      f"{'Speedup':>8} | {'Recall':>8}")
print(f"  {'-'*64}")
for w_fac, w_val, t_l, t_b, rec, pairs in results:
    flag = "✓" if rec >= 0.95 else ("~" if rec >= 0.90 else "✗")
    print(f"  {w_fac:>7.1f}x | {w_val:>5.0f} | {t_l:>8.2f} | {t_b:>8.2f} | "
          f"  {t_b/t_l:>5.1f}x | {rec*100:>7.1f}%{flag}")
print(f"{'='*72}")
print("  → results_W_tuning.txt")

sc.stop()

Spark 4.0.2 | Πυρήνες: 2

Φόρτωση 100,000×1,000 dataset...

Υπολογισμός Baseline (ground truth)...
  Ground truth: 20,477 ζευγάρια (Χρόνος Baseline: 267.25s)

  ΕΛΕΓΧΟΣ: w_factor=0.5x | w=100.0 | ε=200.0 | L=50
  LSH ζευγάρια: 2,800 | Χρόνος LSH: 10.01s | Recall: 13.7% ✗
  Speedup vs baseline: 26.7x

  ΕΛΕΓΧΟΣ: w_factor=1.0x | w=200.0 | ε=200.0 | L=50
  LSH ζευγάρια: 15,578 | Χρόνος LSH: 19.97s | Recall: 76.1% ✗
  Speedup vs baseline: 13.4x

  ΕΛΕΓΧΟΣ: w_factor=1.5x | w=300.0 | ε=200.0 | L=50
  LSH ζευγάρια: 20,243 | Χρόνος LSH: 50.12s | Recall: 98.9% ✓
  Speedup vs baseline: 5.3x

  ΕΛΕΓΧΟΣ: w_factor=2.0x | w=400.0 | ε=200.0 | L=50
  LSH ζευγάρια: 20,470 | Χρόνος LSH: 103.57s | Recall: 100.0% ✓
  Speedup vs baseline: 2.6x

  ΕΛΕΓΧΟΣ: w_factor=3.0x | w=600.0 | ε=200.0 | L=50
  LSH ζευγάρια: 20,477 | Χρόνος LSH: 219.75s | Recall: 100.0% ✓
  Speedup vs baseline: 1.2x

  w_factor |     w |   LSH(s) |  Base(s) |  Speedup |   Recall
  --------------------------------------------------------